# 13 - Macro-Enriched HMM and MoE

This notebook reruns the core pipeline using the enriched feature dataset from notebook 11.

Main upgrade:

- Use `asset_vix_close`
- For US assets, this is US VIX
- For NIFTY, this is India VIX when available

Goal:

> Test whether India VIX and macro-enriched features improve regime detection and all-asset MoE performance.

Input: `../data/processed/features_macro_enriched.csv`  
Outputs:

- `../data/processed/regime_features_macro_enriched.csv`
- `../data/processed/macro_enriched_model_metrics.csv`
- `../data/processed/macro_enriched_summary_metrics.csv`
- `../data/processed/macro_enriched_predictions.csv`
- `../data/processed/macro_enriched_per_asset_metrics.csv`

## Why This Notebook Matters

Earlier, NIFTY used US VIX as the volatility proxy. That is not ideal.

Now we use:

```text
S&P 500 -> US VIX
NASDAQ  -> US VIX
NIFTY   -> India VIX
```

This is a cleaner financial design and a stronger research argument.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from hmmlearn.hmm import GaussianHMM

plt.style.use("seaborn-v0_8-whitegrid")
DATA_DIR = Path("../data/processed")

## 1. Load Macro-Enriched Features

In [2]:
FEATURE_PATH = DATA_DIR / "features_macro_enriched.csv"

features = pd.read_csv(FEATURE_PATH, parse_dates=["Date"])
features = features.sort_values(["ticker", "Date"]).reset_index(drop=True)

print(features.shape)
print(features["ticker"].value_counts())
print(features.columns.tolist())
features.head()

(3694, 30)
ticker
^GSPC    1404
^IXIC    1404
^NSEI     886
Name: count, dtype: int64
['Date', 'ticker', 'Open', 'High', 'Low', 'Close', 'Volume', 'return_1d', 'log_return_1d', 'volatility_7d', 'volatility_14d', 'volatility_30d', 'volatility_60d', 'ma_20_ratio', 'ma_50_ratio', 'ma_200_ratio', 'momentum_7d', 'momentum_14d', 'momentum_30d', 'momentum_60d', 'drawdown', 'high_low_range', 'volume_change', 'vix_close', 'vix_return_1d', 'vix_volatility_30d', 'target_return_1d', 'target_direction_1d', 'india_vix_close', 'asset_vix_close']


,Date,ticker,Open,High,Low,Close,Volume,return_1d,log_return_1d,volatility_7d,...,drawdown,high_low_range,volume_change,vix_close,vix_return_1d,vix_volatility_30d,target_return_1d,target_direction_1d,india_vix_close,asset_vix_close
0,2010-03-30,^GSPC,1173.750000,1177.829956,1168.920044,1173.270020,4.085000e+09,0.000043,0.000043,0.004552,...,-0.000767,0.007622,-0.066409,17.129999,-0.026151,0.033997,-0.003273,0,19.799999,17.129999
1,2010-03-31,^GSPC,1171.750000,1174.560059,1165.770020,1169.430054,4.484340e+09,-0.003273,-0.003278,0.004594,...,-0.004037,0.007540,0.097758,17.590000,0.026854,0.034446,0.007414,1,19.870001,17.590000
2,2010-04-01,^GSPC,1171.229980,1181.430054,1170.689941,1178.099976,4.006870e+09,0.007414,0.007386,0.004654,...,0.000000,0.009174,-0.106475,17.469999,-0.006822,0.033442,0.007928,1,17.620001,17.469999
3,2010-04-05,^GSPC,1178.709961,1187.729980,1178.709961,1187.439941,3.881620e+09,0.007928,0.007897,0.004543,...,0.000000,0.007652,-0.031259,17.020000,-0.025758,0.033352,0.001684,1,17.170000,17.020000
4,2010-04-06,^GSPC,1186.010010,1191.800049,1182.770020,1189.439941,4.086180e+09,0.001684,0.001683,0.004200,...,0.000000,0.007635,0.052700,16.230000,-0.046416,0.034203,-0.005877,0,17.200001,16.230000


## 2. Choose HMM Feature Set

We replace `vix_close` with `asset_vix_close`.

For NIFTY, this should now use India VIX.

In [3]:
hmm_feature_cols = [
    "return_1d",
    "volatility_30d",
    "momentum_30d",
    "ma_50_ratio",
    "drawdown",
    "asset_vix_close",
]

missing_cols = [col for col in hmm_feature_cols if col not in features.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

features[hmm_feature_cols].isna().sum()

return_1d             0
volatility_30d        0
momentum_30d          0
ma_50_ratio        2042
drawdown              0
asset_vix_close       0
dtype: int64

## 3. Train HMM Per Asset

Each asset gets its own HMM because regimes are asset-specific.

The state numbers are arbitrary, so we label them after training.

In [4]:
def label_regimes(summary):
    summary = summary.copy()

    crisis_state = summary["annualized_volatility"].idxmax()
    remaining = [state for state in summary.index if state != crisis_state]

    if len(remaining) == 0:
        return {crisis_state: "Crisis"}

    bull_state = summary.loc[remaining, "annualized_return"].idxmax()
    bear_candidates = [state for state in summary.index if state not in [crisis_state, bull_state]]

    labels = {
        crisis_state: "Crisis",
        bull_state: "Bull",
    }

    if bear_candidates:
        labels[bear_candidates[0]] = "Bear"

    return labels


def fit_hmm_for_ticker(df, ticker, feature_cols, n_regimes=3):
    temp = df[df["ticker"] == ticker].copy()
    temp = temp.sort_values("Date").reset_index(drop=True)

    temp["asset_vix_close"] = temp["asset_vix_close"].ffill().bfill()
    temp = temp.dropna(subset=feature_cols + ["target_return_1d", "target_direction_1d"]).reset_index(drop=True)

    if len(temp) < 250:
        print(f"Skipping {ticker}: only {len(temp)} rows")
        return None

    X = StandardScaler().fit_transform(temp[feature_cols])

    model = GaussianHMM(
        n_components=n_regimes,
        covariance_type="full",
        n_iter=1000,
        random_state=42,
    )
    model.fit(X)

    temp["regime"] = model.predict(X)
    probs = model.predict_proba(X)

    for i in range(n_regimes):
        temp[f"regime_{i}_prob"] = probs[:, i]

    summary = (
        temp
        .groupby("regime")
        .agg(
            observations=("regime", "size"),
            annualized_return=("return_1d", lambda x: x.mean() * 252),
            annualized_volatility=("return_1d", lambda x: x.std() * np.sqrt(252)),
            avg_drawdown=("drawdown", "mean"),
            avg_asset_vix=("asset_vix_close", "mean"),
        )
    )

    labels = label_regimes(summary)
    temp["regime_label"] = temp["regime"].map(labels)

    print(ticker, "rows:", len(temp), "labels:", labels)
    return temp

regime_dfs = []

for ticker in features["ticker"].unique():
    result = fit_hmm_for_ticker(features, ticker, hmm_feature_cols)
    if result is not None:
        regime_dfs.append(result)

regime_macro = pd.concat(regime_dfs, ignore_index=True)
regime_macro = regime_macro.sort_values(["ticker", "Date"]).reset_index(drop=True)

print(regime_macro.shape)
print(regime_macro["ticker"].value_counts())
print(regime_macro["regime_label"].value_counts())
regime_macro.head()

^GSPC rows: 623 labels: {np.int64(2): 'Crisis', np.int64(1): 'Bull', 0: 'Bear'}
^IXIC rows: 623 labels: {np.int64(1): 'Crisis', np.int64(2): 'Bull', 0: 'Bear'}
^NSEI rows: 406 labels: {np.int64(0): 'Crisis', np.int64(2): 'Bull', 1: 'Bear'}
(1652, 35)
ticker
^GSPC    623
^IXIC    623
^NSEI    406
Name: count, dtype: int64
regime_label
Bear      1025
Bull       385
Crisis     242
Name: count, dtype: int64


,Date,ticker,Open,High,Low,Close,Volume,return_1d,log_return_1d,volatility_7d,...,vix_volatility_30d,target_return_1d,target_direction_1d,india_vix_close,asset_vix_close,regime,regime_0_prob,regime_1_prob,regime_2_prob,regime_label
0,2010-04-27,^GSPC,1209.920044,1211.380005,1181.619995,1183.709961,7.454540e+09,-0.023382,-0.023660,0.010809,...,0.072695,0.006463,1,19.820000,22.809999,1,5.500702e-12,1.000000,1.553645e-11,Bull
1,2010-04-28,^GSPC,1184.589966,1195.050049,1181.810059,1191.359985,6.342310e+09,0.006463,0.006442,0.010996,...,0.074230,0.012943,1,22.730000,21.080000,0,9.833277e-01,0.016623,4.970221e-05,Bear
2,2010-04-29,^GSPC,1193.300049,1209.359985,1193.300049,1206.780029,6.059410e+09,0.012943,0.012860,0.011771,...,0.077605,-0.016648,0,21.190001,18.440001,0,9.994668e-01,0.000531,1.791272e-06,Bear
3,2010-04-30,^GSPC,1206.770020,1207.989990,1186.319946,1186.689941,6.048260e+09,-0.016648,-0.016788,0.013373,...,0.084855,0.013121,1,20.350000,22.049999,0,9.998941e-01,0.000065,4.078930e-05,Bear
4,2010-05-03,^GSPC,1188.579956,1205.130005,1188.579956,1202.260010,4.938050e+09,0.013121,0.013035,0.014557,...,0.086658,-0.023838,0,22.139999,20.190001,0,9.986716e-01,0.000053,1.275341e-03,Bear


In [5]:
regime_output_path = DATA_DIR / "regime_features_macro_enriched.csv"
regime_macro.to_csv(regime_output_path, index=False)
print("Saved:", regime_output_path)

Saved: ..\data\processed\regime_features_macro_enriched.csv


## 4. Regime Summary

In [6]:
regime_summary = (
    regime_macro
    .groupby(["ticker", "regime_label"])
    .agg(
        observations=("regime_label", "size"),
        annualized_return=("return_1d", lambda x: x.mean() * 252),
        annualized_volatility=("return_1d", lambda x: x.std() * np.sqrt(252)),
        avg_drawdown=("drawdown", "mean"),
        worst_drawdown=("drawdown", "min"),
        avg_asset_vix=("asset_vix_close", "mean"),
    )
)

regime_summary

observations  annualized_return  annualized_volatility  \
ticker regime_label                                                           
^GSPC  Bear                   412           0.059630               0.112719   
       Bull                   118           0.239130               0.191697   
       Crisis                  93          -0.065461               0.256518   
^IXIC  Bear                   415           0.044405               0.148133   
       Bull                   128           0.461765               0.252683   
       Crisis                  80          -0.217738               0.255740   
^NSEI  Bear                   198          -0.016164               0.107460   
       Bull                   139           0.473948               0.135683   
       Crisis                  69          -0.129293               0.214870   

                     avg_drawdown  worst_drawdown  avg_asset_vix  
ticker regime_label                                               
^GSPC  Bear             -0.014613       -0.062282      15.315607  
       Bull             -0.095237       -0.177210      23.779322  
       Crisis           -0.099125       -0.186753      26.606129  
^IXIC  Bear             -0.024814       -0.110095      15.421542  
       Bull             -0.070786       -0.150348      28.053437  
       Crisis           -0.183632       -0.313433      19.994750  
^NSEI  Bear             -0.043532       -0.139712      15.058838  
       Bull             -0.074049       -0.174838      19.460144  
       Crisis           -0.066799       -0.159783      18.625362

## 5. Build All-Asset Modeling Dataset

We include ticker dummies so the model can distinguish assets.

In [7]:
base_feature_cols = [
    "return_1d",
    "volatility_30d",
    "momentum_30d",
    "ma_50_ratio",
    "drawdown",
    "asset_vix_close",
    "regime_0_prob",
    "regime_1_prob",
    "regime_2_prob",
]

# Include macro columns if they exist and have enough data.
optional_macro_cols = [
    "fed_funds_rate",
    "us_cpi",
    "us_unemployment",
    "us_10y_yield",
    "us_gdp",
    "india_cpi",
    "india_repo_rate",
    "india_gdp_growth",
    "india_10y_yield",
    "usd_inr",
]

usable_macro_cols = []
for col in optional_macro_cols:
    if col in regime_macro.columns and regime_macro[col].notna().sum() > 300:
        usable_macro_cols.append(col)

print("Usable macro columns:", usable_macro_cols)

target_col = "target_direction_1d"
return_col = "target_return_1d"

model_df = regime_macro.replace([np.inf, -np.inf], np.nan).copy()
model_df = model_df.dropna(subset=base_feature_cols + [target_col, return_col, "regime_label", "ticker"]).reset_index(drop=True)
model_df[target_col] = model_df[target_col].astype(int)

# Forward-fill macro columns by ticker if present.
for col in usable_macro_cols:
    model_df[col] = model_df.groupby("ticker")[col].ffill().bfill()

model_df = model_df.dropna(subset=usable_macro_cols).reset_index(drop=True)

ticker_dummies = pd.get_dummies(model_df["ticker"], prefix="ticker", dtype=int)
model_df = pd.concat([model_df, ticker_dummies], axis=1)

feature_cols = base_feature_cols + usable_macro_cols + list(ticker_dummies.columns)

print(model_df.shape)
print(feature_cols)
model_df[["Date", "ticker", "regime_label", target_col] + feature_cols].head()

Usable macro columns: []
(1652, 38)
['return_1d', 'volatility_30d', 'momentum_30d', 'ma_50_ratio', 'drawdown', 'asset_vix_close', 'regime_0_prob', 'regime_1_prob', 'regime_2_prob', 'ticker_^GSPC', 'ticker_^IXIC', 'ticker_^NSEI']


,Date,ticker,regime_label,target_direction_1d,return_1d,volatility_30d,momentum_30d,ma_50_ratio,drawdown,asset_vix_close,regime_0_prob,regime_1_prob,regime_2_prob,ticker_^GSPC,ticker_^IXIC,ticker_^NSEI
0,2010-04-27,^GSPC,Bull,1,-0.023382,0.007319,0.028857,0.020765,-0.027578,22.809999,5.500702e-12,1.000000,1.553645e-11,1,0,0
1,2010-04-28,^GSPC,Bear,1,0.006463,0.007280,0.027513,0.025655,-0.021293,21.080000,9.833277e-01,0.016623,4.970221e-05,1,0,0
2,2010-04-29,^GSPC,Bear,0,0.012943,0.007556,0.034788,0.037015,-0.008626,18.440001,9.994668e-01,0.000531,1.791272e-06,1,0,0
3,2010-04-30,^GSPC,Bear,1,-0.016648,0.008225,0.017893,0.018352,-0.025130,22.049999,9.998941e-01,0.000065,4.078930e-05,1,0,0
4,2010-05-03,^GSPC,Bear,0,0.013121,0.008458,0.036520,0.030068,-0.012339,20.190001,9.986716e-01,0.000053,1.275341e-03,1,0,0


## 6. Define Walk-Forward Windows

In [8]:
unique_dates = np.array(sorted(model_df["Date"].unique()))

initial_train_dates = int(len(unique_dates) * 0.60)
test_window_dates = 125
step_dates = 125

windows = []
start = initial_train_dates

while start + test_window_dates <= len(unique_dates):
    train_dates = unique_dates[:start]
    test_dates = unique_dates[start:start + test_window_dates]
    windows.append((train_dates, test_dates))
    start += step_dates

print("Unique dates:", len(unique_dates))
print("Folds:", len(windows))

for i, (train_dates, test_dates) in enumerate(windows, start=1):
    print(i, pd.Timestamp(train_dates[0]).date(), "to", pd.Timestamp(train_dates[-1]).date(), "|", pd.Timestamp(test_dates[0]).date(), "to", pd.Timestamp(test_dates[-1]).date())

Unique dates: 1029
Folds: 3
1 2010-04-27 to 2019-02-11 | 2019-04-30 to 2020-06-12
2 2010-04-27 to 2020-06-12 | 2020-06-15 to 2022-05-25
3 2010-04-27 to 2022-05-25 | 2022-05-26 to 2025-05-22


## 7. Model And Evaluation Helpers

In [9]:
def make_random_forest():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestClassifier(
            n_estimators=400,
            max_depth=5,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ))
    ])


def make_hist_gradient_boosting():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.03,
            max_leaf_nodes=15,
            l2_regularization=0.1,
            random_state=42,
        ))
    ])


def annualized_sharpe(returns, periods_per_year=252):
    returns = pd.Series(returns).dropna()
    if returns.std() == 0:
        return 0
    return returns.mean() / returns.std() * np.sqrt(periods_per_year)


def max_drawdown(returns):
    returns = pd.Series(returns).fillna(0)
    equity = (1 + returns).cumprod()
    peak = equity.cummax()
    dd = equity / peak - 1
    return dd.min()


def strategy_returns(actual_returns, up_prob, threshold=0.45):
    signal = (np.asarray(up_prob) >= threshold).astype(int)
    return signal * np.asarray(actual_returns)


def evaluate_predictions(model_name, routing, fold, y_true, up_prob, actual_returns, threshold=0.45):
    y_pred = (np.asarray(up_prob) >= threshold).astype(int)
    strat_returns = strategy_returns(actual_returns, up_prob, threshold)

    result = {
        "model": model_name,
        "routing": routing,
        "fold": fold,
        "rows": len(y_true),
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "strategy_sharpe": annualized_sharpe(strat_returns),
        "strategy_max_drawdown": max_drawdown(strat_returns),
        "avg_daily_strategy_return": np.mean(strat_returns),
        "trade_rate": y_pred.mean(),
    }

    try:
        result["roc_auc"] = roc_auc_score(y_true, up_prob)
    except ValueError:
        result["roc_auc"] = np.nan

    return result


def predict_up_probability(model, X):
    return model.predict_proba(X)[:, 1]

model_factories = {
    "Random Forest": make_random_forest,
    "HistGradientBoosting": make_hist_gradient_boosting,
}

## 8. Run Macro-Enriched All-Asset MoE

In [ ]:
threshold = 0.45
min_rows_per_expert = 75

all_metrics = []
all_predictions = []

for model_name, factory in model_factories.items():
    print("\nModel:", model_name)

    for fold_id, (train_dates, test_dates) in enumerate(windows, start=1):
        train_df = model_df[model_df["Date"].isin(train_dates)].copy()
        test_df = model_df[model_df["Date"].isin(test_dates)].copy()

        X_train = train_df[feature_cols]
        y_train = train_df[target_col]
        X_test = test_df[feature_cols]
        y_test = test_df[target_col]

        baseline = factory()
        baseline.fit(X_train, y_train)
        baseline_prob = predict_up_probability(baseline, X_test)

        experts = {}

        for regime_label in sorted(train_df["regime_label"].unique()):
            regime_train = train_df[train_df["regime_label"] == regime_label].copy()

            if len(regime_train) < min_rows_per_expert or regime_train[target_col].nunique() < 2:
                experts[regime_label] = baseline
                continue

            expert = factory()
            expert.fit(regime_train[feature_cols], regime_train[target_col])
            experts[regime_label] = expert

        hard_prob = np.zeros(len(test_df))
        soft_prob = np.zeros(len(test_df))
        test_reset = test_df.reset_index(drop=True)

        regime_to_state = (
            train_df[["regime", "regime_label"]]
            .drop_duplicates()
            .set_index("regime_label")["regime"]
            .to_dict()
        )

        for i, row in test_reset.iterrows():
            x_row = pd.DataFrame([row[feature_cols].values], columns=feature_cols)
            current_regime = row["regime_label"]

            hard_expert = experts.get(current_regime, baseline)
            hard_prob[i] = predict_up_probability(hard_expert, x_row)[0]

            weighted_prob = 0
            total_weight = 0

            for expert_label, expert in experts.items():
                if expert_label not in regime_to_state:
                    continue

                state_number = regime_to_state[expert_label]
                prob_col = f"regime_{state_number}_prob"
                weight = row[prob_col]
                expert_prob = predict_up_probability(expert, x_row)[0]
                weighted_prob += weight * expert_prob
                total_weight += weight

            soft_prob[i] = weighted_prob / total_weight if total_weight > 0 else baseline_prob[i]

        actual_returns = test_df[return_col]

        all_metrics.append(evaluate_predictions(model_name, "Baseline", fold_id, y_test, baseline_prob, actual_returns, threshold))
        all_metrics.append(evaluate_predictions(model_name, "Hard MoE", fold_id, y_test, hard_prob, actual_returns, threshold))
        all_metrics.append(evaluate_predictions(model_name, "Soft MoE", fold_id, y_test, soft_prob, actual_returns, threshold))

        pred_df = test_df[["Date", "ticker", "Close", "regime_label", return_col, target_col]].copy()
        pred_df["model"] = model_name
        pred_df["fold"] = fold_id
        pred_df["baseline_up_prob"] = baseline_prob
        pred_df["hard_moe_up_prob"] = hard_prob
        pred_df["soft_moe_up_prob"] = soft_prob
        all_predictions.append(pred_df)

        print(f"  Finished fold {fold_id}/{len(windows)}")

macro_metrics = pd.DataFrame(all_metrics)
macro_predictions = pd.concat(all_predictions, ignore_index=True)

macro_metrics.head()


Model: Random Forest
  Finished fold 1/3


## 9. Summarize Results

In [ ]:
macro_summary = (
    macro_metrics
    .groupby(["model", "routing"])
    .agg(
        folds=("fold", "count"),
        avg_accuracy=("accuracy", "mean"),
        avg_precision=("precision", "mean"),
        avg_recall=("recall", "mean"),
        avg_f1=("f1", "mean"),
        avg_roc_auc=("roc_auc", "mean"),
        avg_strategy_sharpe=("strategy_sharpe", "mean"),
        avg_strategy_max_drawdown=("strategy_max_drawdown", "mean"),
        avg_daily_strategy_return=("avg_daily_strategy_return", "mean"),
        avg_trade_rate=("trade_rate", "mean"),
    )
    .sort_values("avg_strategy_sharpe", ascending=False)
)

macro_summary

## 10. Per-Asset Results

In [ ]:
def evaluate_asset_group(group, prob_col, threshold=0.45):
    y_true = group[target_col]
    probs = group[prob_col]
    y_pred = (probs >= threshold).astype(int)
    strat = strategy_returns(group[return_col], probs, threshold)

    return pd.Series({
        "rows": len(group),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "strategy_sharpe": annualized_sharpe(strat),
        "max_drawdown": max_drawdown(strat),
        "avg_daily_return": np.mean(strat),
        "trade_rate": y_pred.mean(),
    })

per_asset_results = []

for model_name in macro_predictions["model"].unique():
    temp = macro_predictions[macro_predictions["model"] == model_name].copy()

    for routing, prob_col in [
        ("Baseline", "baseline_up_prob"),
        ("Hard MoE", "hard_moe_up_prob"),
        ("Soft MoE", "soft_moe_up_prob"),
    ]:
        asset_result = temp.groupby("ticker").apply(lambda g: evaluate_asset_group(g, prob_col, threshold)).reset_index()
        asset_result["model"] = model_name
        asset_result["routing"] = routing
        per_asset_results.append(asset_result)

macro_per_asset = pd.concat(per_asset_results, ignore_index=True)
macro_per_asset.sort_values(["ticker", "strategy_sharpe"], ascending=[True, False])

## 11. Compare Before vs After India VIX

Before: notebook 08 using original `vix_close`.

After: this notebook using `asset_vix_close`.

In [ ]:
before_path = DATA_DIR / "all_asset_summary_metrics.csv"

if before_path.exists():
    before = pd.read_csv(before_path)
    before["version"] = "Before asset_vix_close"
    before = before.rename(columns={"model_family": "model"})

    after = macro_summary.reset_index().copy()
    after["version"] = "After asset_vix_close"

    comparison = pd.concat([
        before[["version", "model", "routing", "avg_accuracy", "avg_f1", "avg_roc_auc", "avg_strategy_sharpe", "avg_strategy_max_drawdown"]],
        after[["version", "model", "routing", "avg_accuracy", "avg_f1", "avg_roc_auc", "avg_strategy_sharpe", "avg_strategy_max_drawdown"]],
    ], ignore_index=True)

    comparison.sort_values("avg_strategy_sharpe", ascending=False)
else:
    print("No before metrics found.")

## 12. Save Outputs

In [ ]:
macro_metrics.to_csv(DATA_DIR / "macro_enriched_model_metrics.csv", index=False)
macro_summary.to_csv(DATA_DIR / "macro_enriched_summary_metrics.csv")
macro_predictions.to_csv(DATA_DIR / "macro_enriched_predictions.csv", index=False)
macro_per_asset.to_csv(DATA_DIR / "macro_enriched_per_asset_metrics.csv", index=False)

print("Saved:", DATA_DIR / "macro_enriched_model_metrics.csv")
print("Saved:", DATA_DIR / "macro_enriched_summary_metrics.csv")
print("Saved:", DATA_DIR / "macro_enriched_predictions.csv")
print("Saved:", DATA_DIR / "macro_enriched_per_asset_metrics.csv")

## 13. Visualize Summary

In [ ]:
plot_df = macro_summary.reset_index()
plot_df["label"] = plot_df["model"] + " - " + plot_df["routing"]
plot_df = plot_df.sort_values("avg_strategy_sharpe", ascending=False)

plt.figure(figsize=(12, 5))
plt.bar(plot_df["label"], plot_df["avg_strategy_sharpe"])
plt.title("Macro-Enriched All-Asset Strategy Sharpe")
plt.ylabel("Average Strategy Sharpe")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Interpretation

Use this notebook to answer:

1. Did India VIX improve NIFTY performance?
2. Did `asset_vix_close` improve the all-asset MoE result?
3. Does the Soft MoE still beat the baseline after the cleaner volatility proxy?

If NIFTY improves, this becomes a strong final research upgrade.